In [1]:
# numpy based vector objects, fixed dtype float64

# record vectors need to have variable length, but numpy won't allow this with np.array .
# we could use a list of numpy arrays but it may be slow to access elements.
# we need to be able to acess elements quickly within a numpy system with variable length, so 
G = 6.67430e-11  # gravitational constant
k = 8.9875517923e9  # Coulomb's constant  # permeability of free space


from typing import List, Union
import numpy as np
import random

from vispy import scene
from vispy.scene import visuals
import multiprocessing as mproc
##########################################################################################
#                                       Particle class                                   #
##########################################################################################


class Particle:
    def __init__(self,
                 pos: np.ndarray, vel: np.ndarray,
                 mass: float, radius: float,
                 charge: float, color: Union[str, np.ndarray] = 'white'):
        
        self.pos = np.array(pos, dtype=float)
        self.vel = np.array(vel, dtype=float)
        self.mass = mass
        self.radius = radius
        self.charge = charge
        self.color = color
    
    
    def __repr__(self) -> str:
        return f"\nParticle(p={self.pos},v={self.vel},m={self.mass},r={self.radius},c={self.charge})"



##########################################################################################
#                                       Node class                                       #
##########################################################################################


class Node:
    def __init__(self, pos: np.ndarray, size: float, mass: float=0., depth=0) -> None:
        self.pos = pos # center of node
        self.size = size # each node is cubic so only need one size value
        self.mass = mass # total mass of node, including children
        self.children = []  # child nodes
        self.particles = []  # particles directly in this node
        self.is_end = False # is this a leaf node
        self.depth = depth # depth of node in tree
        self.cmass = pos # center of mass of node (start at center of node for now)
        #debug print(f"d{depth} Node: pos={self.pos}, size={self.size}, mass={self.mass}")
    
    
    def insert(self, particle: Particle) -> None:
        '''insert particle into node and update mass of node'''
        self.particles.append(particle) 
        self.mass += particle.mass
        #debug print(f"inserted particle with mass {particle.mass} into node with mass {self.mass}")

    
    def split(self) -> None:
        '''split node into 8 octants and redistribute particles to children nodes'''
        if not self.is_end: # if it is already a leaf node, don't split
            #debug print(f"splitting a d{self.depth} Node: pos={self.pos}, size={self.size}, mass={self.mass}")
            #split into 8 octants
            for i in [1,-1]:  # bisecting in each dimension
                for j in [1,-1]:
                    for k in [1,-1]:
                        pos = self.pos + np.array([i, j, k]) * self.size / 2 # calculate child node position
                        self.children.append(Node(pos, self.size/2, depth=self.depth+1))  # add as child node
            
            for particle in self.particles: # redistribute particles to children nodes
                for child in self.children:  
                    if child.contains(particle): # if particle is within child node, insert
                        child.insert(particle)
                        break
                else:
                    raise ValueError(f"particle not in any child node: \n{particle}")
                    
            self.children = [child for child in self.children if child.mass > 0] # get rid of empty nodes
            self.particles = []  # clear particles list after redistribution, since they are now in children
            for child in self.children:
                if len(child.particles) > 1: # if child has more than one particle, split again
                    child.split()
                    self.fast_cmass() # update center of mass of parent node
                else:
                    child.is_end = True # if child has only one particle, it is a leaf node
    
    
    def contains(self, particle: Particle) -> bool:
        """check if particle is within node's volume"""
        return np.all(np.abs(particle.pos - self.pos) <= self.size)  # check if particle is within node's volume by comparing each dimension 
    
    
    def fast_cmass(self) -> np.ndarray:
        """calculate center of mass of node using fast method"""
        self.cmass = np.sum([p.mass*p.pos for p in [*self.particles,*self.children]], axis=0) / self.mass # sum of mass*position / total mass. faster because we remove most children before using it.
        return self.cmass
    
    
    def __repr__(self) -> str: # a debug representation of the node containing its children's and particles' info recursively
        return (f"""
d{self.depth}Node: pos={self.pos} size={self.size} mass={self.mass} 
children({len(self.children)}): {[child for child in self.children]}
particles({len(self.particles)}): {[particle for particle in self.particles]}""")


    def __str__(self) -> str: # human readable representation of the node
        return (f"""d{self.depth}Node: pos={self.pos} size={self.size} mass={self.mass} [{len(self.children)}ch,{len(self.particles)}p]""")
    

    def treeview(self) -> str: # recursive representation of the tree
        """print tree structure of node and children recursively"""
        bars = "   ⎸"*self.depth
        info = f"⟶  d{self.depth} Node[{len(self.children)}ch,{len(self.particles)}p], cmass {self.cmass}, Mass:{self.mass}"
        if not self.is_end:
            info = "\u001b[37m\u001b[1m" + info + "\u001b[37m\u001b[0m"
        print(bars + info)
        for c in self.children:
            c.treeview()

    
    def dist(self, other) -> np.ndarray: 
        """calculate distance between center of masses of two nodes"""
        return np.linalg.norm(self.cmass - other.cmass)

##########################################################################################
#                                        BHTree class                                    #
##########################################################################################

class BHTree:
    def __init__(self, particles, theta:float=0.5, origin:np.ndarray=np.zeros(3), size:float=100.) -> None:
        self.theta = theta
        #debug [np.mean(comp) for comp in zip(*[p.pos for p in particles])]
        #debug initpos = np.array([np.mean(comp) for comp in zip(*[p.pos for p in particles])])
        self.root = Node(
            pos= origin, size = size    #commented out bits are for a case where we might not know the size of the system initially
            #debug pos = initpos,
            #debug size = np.max([np.max([np.abs(particle.pos - initpos)]) for particle in particles])
            )
        for particle in particles:
            self.root.insert(particle) # insert particles into root node 
        self.root.split() # begin recursive splitting of nodes

    def node_of_particle(self, particle:Particle) -> Node:
        '''find the node containing a particle recursively'''
        found_part = False 
        trials = [self.root]
        while not found_part:
            trial = trials.pop(0)
            if trial.contains(particle):
                if trial.is_end:
                    found_part = True
                    return trial
                else:
                    trials.extend(trial.children)
            elif not trials:
                raise ValueError(f"particle not found in any node: {particle}")


    

##########################################################################################
#                                           Main                                         #
########################################################################################## 


# create some particles
random.seed(0)
particles = [
    Particle(
        pos=[random.uniform(-100, 100) for _ in range(3)],  # random position
        vel=[random.uniform(-1, 1) for _ in range(3)],  # random velocity
        mass=random.uniform(1, 1000),  # random mass
        radius=random.uniform(0.1, 1),  # random radius
        charge=random.uniform(-1, 1),  # random charge
        color=(random.random(), random.random(), random.random())  # random color
    ) 
    for _ in range(10000)
]

tree = BHTree(particles)
tree.root.treeview()




⟶  d0 Node[8ch,0p], cmass [ 0.12006441 -0.63497945 -0.05050606], Mass:4991069.539574348
   ⎸⟶  d1 Node[8ch,0p], cmass [48.68801772 51.79028914 51.46269153], Mass:607461.5812900056
   ⎸   ⎸⟶  d2 Node[8ch,0p], cmass [77.58479916 75.38370781 77.37033202], Mass:80884.06834538784
   ⎸   ⎸   ⎸⟶  d3 Node[8ch,0p], cmass [88.01614247 89.60366691 87.22445461], Mass:17206.04795020047
   ⎸   ⎸   ⎸   ⎸⟶  d4 Node[5ch,0p], cmass [93.28820022 91.26383473 93.51829671], Mass:4808.016292024411
   ⎸   ⎸   ⎸   ⎸   ⎸⟶  d5 Node[0ch,1p], cmass [96.875 96.875 96.875], Mass:491.44444368279363
   ⎸   ⎸   ⎸   ⎸   ⎸⟶  d5 Node[0ch,1p], cmass [96.875 90.625 96.875], Mass:775.9971390899278
   ⎸   ⎸   ⎸   ⎸   ⎸⟶  d5 Node[0ch,1p], cmass [96.875 90.625 90.625], Mass:781.3120217987477
   ⎸   ⎸   ⎸   ⎸   ⎸⟶  d5 Node[2ch,0p], cmass [90.625 90.625 96.875], Mass:958.3212526514188
   ⎸   ⎸   ⎸   ⎸   ⎸   ⎸⟶  d6 Node[0ch,1p], cmass [89.0625 92.1875 98.4375], Mass:67.463237345432
   ⎸   ⎸   ⎸   ⎸   ⎸   ⎸⟶  d6 Node[0ch,1p], cmass

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "e:\Repositories\nbody\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "e:\Repositories\nbody\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "e:\Repositories\nbody\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "e:\Repositories\nbody\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "D:\Python\Python311\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "D:\Python\Python311\Lib\asyncio\base_events.py", line 1936, in _run_once
    handle._run()
  File "D:\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
  File "e:\Repositories\nbody\.venv\Lib\site-pac

: 

In [ ]:
# -*- coding: utf-8 -*-
# Copyright (c) Vispy Development Team. All Rights Reserved.
# Distributed under the (new) BSD License. See LICENSE.txt for more info.
# vispy: gallery 2
"""
Plot different styles of ColorBar
=================================
"""

from vispy import plot as vp
import numpy as np
%gui qt

# arg( e^(1/z) )
def exp_z_inv(x, y):
    z = complex(x, y)
    f = np.exp(1.0 / z)
    return np.angle(f, deg=True)


# create a 2d grid whose elements are of exp_z_inv
def gen_image(width, height):
    x_vals = np.linspace(-0.5, 0.5, width)
    y_vals = np.linspace(-0.5, 0.5, height)

    grid = np.meshgrid(x_vals, y_vals)
    v_fn = np.vectorize(exp_z_inv)

    return v_fn(*grid).astype(np.float32)

fig = vp.Fig(size=(800, 600), show=False)
plot = fig[0, 0]
plot.bgcolor = "#efefef"

img = gen_image(500, 500)
plot.image(img, cmap="hsl")
plot.camera.set_range((100, 400), (100, 400))

positions = ["top", "bottom", "left", "right"]

for position in positions:
    plot.colorbar(position=position,
                  label="argument of e^(1/z)",
                  clim=("0°", "180°"),
                  cmap="hsl",
                  border_width=1,
                  border_color="#aeaeae")
if __name__ == '__main__':
    fig.show(run=True)

RFBOutputContext()